## Dataset Porto Seguro Safe Driver Prediction

In [1]:
import pandas as pd
df_train = pd.read_csv('../Cleaned-Dataset/train.csv')
df_test = pd.read_csv('../Cleaned-Dataset/test.csv')

In [2]:
train_samples = int(df_train.shape[0] * 0.8)

X_train, y_train = df_train[:train_samples].drop(columns=['target']), df_train[:train_samples]['target']
X_valid, y_valid = df_train[train_samples:].drop(columns=['target']), df_train[train_samples:]['target']

test = df_test

In [3]:
# Encoding 
# Target encode only high-cardinality column
cat_cols_target = ["ps_car_11_cat"]

cat_cols_final = [col for col in X_train.columns if col.endswith("_cat")]
cat_cols_onehot = [col for col in cat_cols_final if col not in cat_cols_target]

from category_encoders import TargetEncoder, OneHotEncoder
# --- Target Encoding ---
target_encoder = TargetEncoder(cols=cat_cols_target, smoothing=0.3)

X_train = target_encoder.fit_transform(X_train, y_train)
X_valid = target_encoder.transform(X_valid)
test = target_encoder.transform(test)

# --- One Hot Encoding ---
onehot_encoder = OneHotEncoder(cols=cat_cols_onehot, use_cat_names=True)

X_train = onehot_encoder.fit_transform(X_train)
X_valid = onehot_encoder.transform(X_valid)
test = onehot_encoder.transform(test)

- optimal max_depth = 6, min_samples_split = 50 for decision Tree

## Model-1: Random Forest From Scratch with Sklearn Decision Tree


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import mode
from collections import defaultdict

class RandomForestClassifierScratchWithDecisionTreeSklearn:
    def __init__(
        self,
        n_estimators=100,
        max_depth=10,
        min_samples_split=50,
        min_samples_leaf=20,
        max_features='sqrt',
        bootstrap=True,
        oob_score=False,
        random_state=None,
        n_jobs=1,
        criterion='gini',
        class_weight='balanced',
        **tree_kwargs
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features          
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.criterion = criterion
        self.class_weight = class_weight
        self.tree_kwargs = tree_kwargs
        self.trees = []
        self.oob_score_ = None
        self.feature_importances_ = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        n_samples = X.shape[0]
        rng = np.random.RandomState(self.random_state)

        self.trees = []
        oob_predictions = [[] for _ in range(n_samples)]   # list of predictions per sample

        for i in range(self.n_estimators):
            # ====================== BOOTSTRAP SAMPLING ====================== #
            if self.bootstrap:
                sample_idx = rng.choice(n_samples, size=n_samples, replace=True)
                # OOB mask (samples never selected)
                in_bag = np.zeros(n_samples, dtype=bool)
                in_bag[sample_idx] = True
                oob_idx = np.where(~in_bag)[0]
            else:
                sample_idx = np.arange(n_samples)
                oob_idx = np.array([], dtype=int)

            # ====================== BUILD TREE WITH ALL PARAMETERS ====================== #
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf = self.min_samples_leaf,
                max_features=self.max_features,
                criterion=self.criterion,
                class_weight=self.class_weight,
                random_state=rng.randint(0, 2**31 - 1) if self.random_state is not None else None,
                **self.tree_kwargs
            )
            tree.fit(X[sample_idx], y[sample_idx])
            self.trees.append(tree)

            # ====================== OOB PREDICTIONS =================================== #
            if self.oob_score and len(oob_idx) > 0:
                oob_pred = tree.predict_proba(X[oob_idx])[:, 1]
                for j, idx in enumerate(oob_idx):
                    oob_predictions[idx].append(oob_pred[j])
        
        # ================== Compute OOB Score ========================================== #
        if self.oob_score:
            oob_final_pred = np.full(n_samples, -1, dtype=int)
            for i in range(n_samples):
                if len(oob_predictions[i]) > 0:
                    oob_final_pred[i] = mode(oob_predictions[i], keepdims=False).mode
            valid = oob_final_pred != -1
            self.oob_score_ = roc_auc_score(y[valid], oob_final_pred[valid])
        
        # =================== Feature Importances (aggregated) ======================== #
        importances = np.array([tree.feature_importances_ for tree in self.trees])
        self.feature_importances_ = np.mean(importances, axis=0)
        return self
    
    def predict(self, X):
        X = np.array(X)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Majority Vote
        return mode(tree_preds, axis=0, keepdims=False).mode
    
    def predict_proba(self, X):
        X = np.array(X)
        probs = np.array([tree.predict_proba(X) for tree in self.trees])
        return np.mean(probs, axis=0)
    

In [26]:

rf_model_1 = RandomForestClassifierScratchWithDecisionTreeSklearn(
    n_estimators=300,
    max_depth=8,
    min_samples_split=50,
    min_samples_leaf=20,
    max_features=0.5,
    class_weight={0:1, 1:25},
    bootstrap=True,
    oob_score=True,
    random_state=42
) 
 
import time
start = time.time() 
rf_model_1.fit(X_train, y_train)
end = time.time()

print("Model 1 OOB Score: ", rf_model_1.oob_score_)
print("Model 1 (Sklearn DT) Training time: ", (end - start)/60, "mins.")

Model 1 OOB Score:  0.5
Model 1 (Sklearn DT) Training time:  46.595812646547955 mins.


In [28]:
from sklearn.metrics import roc_auc_score

y_preds_model1 = rf_model_1.predict_proba(X_valid)
print("Model-1 AUC Score : ", roc_auc_score(y_valid, y_preds_model1[:, 1]))

Model-1 AUC Score :  0.6221080299220281


In [ ]:
# # --- Test Prediction ---
# test_preds_model1 = rf_model_1.predict_proba(test)[:, 1]

# # --- create submission ---
# submission = pd.DataFrame({
#     "id": test['id'],
#     "target": test_preds_model1.round(4)
# })

In [ ]:
# submission.to_csv("../Final-Submission/RF-submission_model1.csv", index=False)

## Sklearn Model

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

rf_sklearn = RandomForestClassifier(
    n_estimators=500,           # more trees = smoother
    max_depth=8,                # prevent overfitting
    min_samples_split=50,
    min_samples_leaf=20,
    max_features=0.5,        # critical for decorrelation on redundant features
    bootstrap=True,
    oob_score=True,             # we will compare with our OOB
    n_jobs=-1,
    random_state=42,
    criterion='gini',
    class_weight= {0:1, 1:25}  # 'balanced'     # handles mild imbalance
)


In [ ]:
rf_sklearn.fit(X_train, y_train)
print("Sklearn OOB score:", rf_sklearn.oob_score_)
# 10 mins

Sklearn OOB score: 0.731964911617514


In [22]:
from sklearn.metrics import roc_auc_score

y_preds_sklearn = rf_sklearn.predict_proba(X_valid)[:, 1]
print("valid AUC Score:", roc_auc_score(y_valid, y_preds_sklearn))
print("Top 5 features (should down-weight redundant & noise):", 
      np.argsort(rf_sklearn.feature_importances_)[-5:])

valid AUC Score: 0.6231987076019523
Top 5 features (should down-weight redundant & noise): [ 9 30 26 83 86]


## Models From Scratch 

### Decision Tree From Scratch

In [ ]:
import numpy as np
import pandas as pd

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, class_label=None, proba=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.class_label = class_label
        self.proba = proba

class DecisionTreeFromScratch_Opt_Time:
    def __init__(self, max_depth=10, min_samples_leaf=10, min_samples_split=20, 
                 max_features=None, pos_weight=5, reg_gamma=0.0, reg_lambda=0.1, 
                 min_impurity_decrease=1e-4):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split  = min_samples_split
        self.max_features = max_features
        self.pos_weight = pos_weight

        # Added more regularization parameters
        self.min_impurity_decrease = min_impurity_decrease
        self.reg_lambda = reg_lambda                # L2 regularization
        self.reg_gamma = reg_gamma                 # feature split penalty
        self.root = None
    
    def _best_split(self, X, y, indices):
        m = len(indices)

        if m < self.min_samples_split:
            return None, None

        best_gini = float('inf')
        best_feature, best_threshold = None, None

        n_features = X.shape[1]

        # Feature sampling
        features = np.arange(n_features)
        if self.max_features:
            max_feats = int(self.max_features)
            features = np.random.choice(n_features, max_feats, replace=False)

        for feature in features:
            feature = int(feature)

            # Sort once
            sorted_idx = indices[np.argsort(X[indices, feature])]
            X_sorted = X[sorted_idx, feature]
            y_sorted = y[sorted_idx]

            # Convert labels to 0/1 (important for speed)
            y_bin = (y_sorted == 1).astype(np.int32)

            # Total counts
            total_pos = y_bin.sum()
            total_neg = m - total_pos

            # Cumulative sums → vectorized left counts
            left_pos = np.cumsum(y_bin)[:-1]
            left_neg = np.cumsum(1 - y_bin)[:-1]

            # Right counts
            right_pos = total_pos - left_pos
            right_neg = total_neg - left_neg

            # Valid splits mask
            valid = (
                (np.arange(1, m) >= self.min_samples_leaf) &
                ((m - np.arange(1, m)) >= self.min_samples_leaf) &
                (X_sorted[1:] != X_sorted[:-1])
            )

            if not np.any(valid):
                continue

            # Gini computation (vectorized)
            left_total = left_pos + left_neg
            right_total = right_pos + right_neg

            # Use weighted positive for gini calculation
            left_pos_w = left_pos * self.pos_weight
            left_total_w = left_total * self.pos_weight

            right_pos_w = right_pos * self.pos_weight
            right_total_w = right_total * self.pos_weight

            # Avoid division by zero
            left_total = np.maximum(left_total_w, 1)
            right_total = np.maximum(right_total_w, 1)

            # Probabilities
            p_left_pos = left_pos_w / left_total_w
            p_left_neg = left_neg / left_total_w

            p_right_pos = right_pos_w / right_total_w
            p_right_neg = right_neg / right_total_w

            # Gini 
            gini_left = 1 - (p_left_pos**2 + p_left_neg**2)
            gini_right = 1 - (p_right_pos**2 + p_right_neg**2)

            # Weighted gini + gamma penalty
            weights_left = left_total / m
            weights_right = right_total / m

            weighted_gini = weights_left * gini_left + weights_right * gini_right + self.reg_gamma

            # Apply valid mask
            weighted_gini[~valid] = np.inf

            # Find best split for this feature
            idx = np.argmin(weighted_gini)
            gini = weighted_gini[idx]

            if gini < best_gini:
                best_gini = gini
                best_feature = feature
                best_threshold = (X_sorted[idx] + X_sorted[idx + 1]) / 2

            # Early stopping for already optimized gain
            if best_gini < 1e-6:
                break

        return best_feature, best_threshold

    # Compute child impurity
    def gini_calc(y_sub):
        if len(y_sub) == 0:
            return 0
        p = np.mean(y_sub)
        return 1 - (p**2 + (1 - p)**2)

    # Recursive tree building (No Array Copying)
    def _build_tree(self, X, y, indices, depth=0):
        """Recursive tree building."""
        m = len(indices)
        y_subset = y[indices]

        pos = np.sum(y_subset == 1)
        neg = m - pos

        # laplace smoothing
        proba = (pos + self.reg_lambda) / (pos + neg + self.reg_lambda)

        # Compute parent gini (for pruning)
        p = pos / m if m > 0 else 0
        parent_gini = 1 - (p**2 + (1-p)**2)

        # stopping condition.
        if depth >= self.max_depth or m < self.min_samples_split or len(np.unique(y_subset)) == 1:
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        
        feature, threshold = self._best_split(X, y, indices)

        if feature is None: 
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        
        left_indices = indices[X[indices, feature] <= threshold]
        right_indices = indices[X[indices, feature] > threshold]

        # Check min impurity decrease
        left_gini = self.gini_calc(y[left_indices])
        right_gini = self.gini_calc(y[right_indices])

        child_gini = (len(left_indices)/m)*left_gini + (len(right_indices)/m)*right_gini

        if (parent_gini - child_gini) < self.min_impurity_decrease:
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        

        left = self._build_tree(X, y, left_indices, depth + 1)
        right = self._build_tree(X, y, right_indices, depth + 1)
        return Node(feature=feature, threshold=threshold, left=left, right=right)
    
    # Fit
    def fit(self, X, y):
        """Fit the model."""
        X = np.array(X, dtype=np.float32)
        y = (np.array(y).ravel() == 1).astype(np.int32)
        indices = np.arange(len(y))
        self.root = self._build_tree(X, y, indices)
        return self

    def _predict_one(self, x, node):
        """Traverse tree for one sample."""
        if node.class_label is not None:
            return node.class_label
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)
        
    def predict(self, X):
        """Predict the model for a test case."""
        X = np.array(X)
        return np.array([self._predict_one(x, self.root) for x in X])

    def _predict_proba_one(self, x, node):
        """Traverse tree for one sample to get target probability."""
        if node.proba is not None:
            return node.proba
        if x[node.feature] <= node.threshold:
            return self._predict_proba_one(x, node.left)
        else:
            return self._predict_proba_one(x, node.right)
        
    def predict_proba(self, X):
        """Predict the probability of target with model for a test case."""
        X = np.array(X)
        return np.array([self._predict_proba_one(x, self.root) for x in X])

### Random Forest From Scratch

In [30]:
class RandomForestClassifierScratch:

    def __init__(
        self,
        n_estimators=100,
        max_depth=10,
        min_samples_split=50,
        max_features=None,
        bootstrap=True,
        oob_score=False,
        pos_weight=5,
        random_state=None
    ):

        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state
        self.pos_weight = pos_weight
        
        self.trees = []
        self.oob_score_ = None

    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        n_samples = X.shape[0]

        rng = np.random.RandomState(self.random_state)

        self.trees = []

        oob_predictions = [[] for _ in range(n_samples)]

        # ====================================== TRAIN DECISION TREES =========================== #
        for i in range(self.n_estimators):

            # ============================== BOOTSTRAP SAMPLING ================================= #
            if self.bootstrap:
                sample_idx = rng.choice(n_samples, n_samples, replace=True)

                in_bag = np.zeros(n_samples, dtype=bool)
                in_bag[sample_idx] = True

                oob_idx = np.where(~in_bag)[0]
            else:
                sample_idx = np.arange(n_samples)
                oob_idx = np.array([], dtype=int)

            tree = DecisionTreeFromScratch_Opt_Time(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                pos_weight = self.pos_weight
            )

            tree.fit(X[sample_idx], y[sample_idx])

            self.trees.append(tree)

            # ========================= PREDICT OOB SAMPLE ==================================== #
            if self.oob_score and len(oob_idx) > 0:

                oob_pred = tree.predict_proba(X[oob_idx])
                if (i+1) % 20 == 0:
                    print(f"Trained {i+1}/{self.n_estimators} tree. OOB Prediction: {roc_auc_score(y[oob_idx], oob_pred)}")
                
                for j, idx in enumerate(oob_idx):
                    oob_predictions[idx].append(oob_pred[j])

        # ======================= COMPUTE OOB SCORE ==================================== #
        if self.oob_score:

            oob_final_pred = np.full(n_samples, -1)

            for i in range(n_samples):

                if len(oob_predictions[i]) > 0:

                    oob_final_pred[i] = mode(oob_predictions[i], keepdims=False).mode

            valid = oob_final_pred != -1

            self.oob_score_ = roc_auc_score(y[oob_idx], oob_pred)

        return self

    # ----------------------------
    # Predict
    # ----------------------------
    def predict(self, X):

        X = np.array(X)

        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        return mode(tree_preds, axis=0, keepdims=False).mode.ravel()

    # ----------------------------
    # Predict Probabilities
    # ----------------------------
    def predict_proba(self, X):

        X = np.array(X)

        probs = np.array([tree.predict_proba(X) for tree in self.trees])

        return np.mean(probs, axis=0)


### Model Train 

### Model-2

In [ ]:
rf_model_2 = RandomForestClassifierScratch(
    n_estimators=200,
    max_depth=8,
    min_samples_split=50,
    max_features=int(0.5 * X_train.shape[1]),
    bootstrap=True,
    oob_score=True,
    random_state=42
)

In [32]:
import time

start = time.time()
rf_model_2.fit(X_train, y_train)
end = time.time()

print("Model 2 OOB Score: ", rf_model_2.oob_score_)
print(f"RF Model Training Time: ", (end - start)/60, "mins.")

Trained 20/200 tree. OOB Prediction: 0.5974193933105394
Trained 40/200 tree. OOB Prediction: 0.5875024925112291
Trained 60/200 tree. OOB Prediction: 0.5932395645554522
Trained 80/200 tree. OOB Prediction: 0.5848200628316593
Trained 100/200 tree. OOB Prediction: 0.5833774978525087
Trained 120/200 tree. OOB Prediction: 0.5936318379806457
Trained 140/200 tree. OOB Prediction: 0.5952611396316635
Trained 160/200 tree. OOB Prediction: 0.5919012628612136
Trained 180/200 tree. OOB Prediction: 0.5924564870368539
Trained 200/200 tree. OOB Prediction: 0.5879904823278403
Model 2 OOB Score:  0.5879904823278403
RF Model Training Time:  27.84320183197657 mins.


In [33]:
from sklearn.metrics import roc_auc_score

y_preds_model2 = rf_model_2.predict_proba(X_valid)
auc_model2 = roc_auc_score(y_valid, y_preds_model2)
gini_model2 = 2 * auc_model2 - 1
print(auc_model2)
print(gini_model2)

0.6251191993551389
0.25023839871027787


### Model-3

In [31]:
rf_model_3 = RandomForestClassifierScratch(
    n_estimators=300,
    max_depth=8,
    min_samples_split=30,
    max_features=int(0.2 * X_train.shape[1]),   #int(np.sqrt(X_train.shape[1])),
    bootstrap=True,
    oob_score=True,
    random_state=42
)

In [32]:
import time

start = time.time()
rf_model_3.fit(X_train, y_train)
end = time.time()

print("Model 3 OOB Score: ", rf_model_3.oob_score_)
print(f"RF Model Training Time: ", (end - start)/60, "mins.")

Trained 20/300 tree. OOB Prediction: 0.9606989394770524
Trained 40/300 tree. OOB Prediction: 0.96087239806591
Trained 60/300 tree. OOB Prediction: 0.9612839556878986
Trained 80/300 tree. OOB Prediction: 0.9611355380290043
Trained 100/300 tree. OOB Prediction: 0.9598136550220085
Trained 120/300 tree. OOB Prediction: 0.9620290647350918
Trained 140/300 tree. OOB Prediction: 0.9619995657093224
Trained 160/300 tree. OOB Prediction: 0.9610899394605235
Trained 180/300 tree. OOB Prediction: 0.9608210126278909
Trained 200/300 tree. OOB Prediction: 0.961356881572931
Trained 220/300 tree. OOB Prediction: 0.9614597856147007
Trained 240/300 tree. OOB Prediction: 0.96124323428382
Trained 260/300 tree. OOB Prediction: 0.9621198689094663
Trained 280/300 tree. OOB Prediction: 0.9612615701062736
Trained 300/300 tree. OOB Prediction: 0.9613107641366712
Model 3 OOB Score:  0.9635885578439588
RF Model Training Time:  263.79294603268306 mins.


In [ ]:
from sklearn.metrics import roc_auc_score

y_preds_model3 = rf_model_3.predict_proba(X_valid)
auc_model3 = roc_auc_score(y_valid, y_preds_model3)
gini_model3 = 2 * auc_model3 - 1
print(f"AUC Score for Model-3 :", auc_model3)
print(f"Gini Score for Model-3 :", gini_model3)

AUC Score for Model-3 : 0.6225333336458048
Gini Score for Model-3 : 0.24506666729160953


### Summary

> - Typical good models:
>   - AUC: ~0.63 – 0.65
>   - Gini: ~0.26 – 0.30

#### Model - 2
> - n = 100, depth = 6, split=50, oob=true, features=srqt(X.shape[1]) 
>   - AUC: 0.6242
>   - Gini: 0.2485

#### Model - 3
> - n=300, depth=8, split=30, oob=true, features=0.2*X.shape[1]
>   - AUC: 0.6225
>   - Gini: 0.2450


### Best Model Submission

In [ ]:
# # train the best model

# test_preds_model2 = rf_model_2.predict_proba(test)

# submission = pd.DataFrame({
#     "id": test['id'],
#     "target": test_preds_model2.round(4)
# })

# submission.to_csv("../Final-Submission/RF-submission_model2.csv")